# SpaceX Falcon 9 – Data Collection with the SpaceX REST API

In [1]:
import requests, pandas as pd, numpy as np, datetime
pd.set_option('display.max_columns', None)

In [2]:
# Past launches (IBM static copy of https://api.spacexdata.com/v4/launches/past for reproducibility)
static_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'
data = pd.json_normalize(requests.get(static_url).json())
data = data[['rocket','payloads','launchpad','cores','flight_number','date_utc']]
data = data[data['cores'].map(len)==1]; data = data[data['payloads'].map(len)==1]
data['cores'] = data['cores'].map(lambda x: x[0]); data['payloads'] = data['payloads'].map(lambda x: x[0])
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]
data.shape

(94, 7)

In [3]:
API='https://api.spacexdata.com/v4'
BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude=[[] for _ in range(15)]
def getBoosterVersion(d):
    for x in d['rocket']: BoosterVersion.append(requests.get(f'{API}/rockets/{x}').json()['name'])
def getLaunchSite(d):
    for x in d['launchpad']:
        r=requests.get(f'{API}/launchpads/{x}').json(); Longitude.append(r['longitude']); Latitude.append(r['latitude']); LaunchSite.append(r['name'])
def getPayloadData(d):
    for x in d['payloads']:
        r=requests.get(f'{API}/payloads/{x}').json(); PayloadMass.append(r['mass_kg']); Orbit.append(r['orbit'])
def getCoreData(d):
    for core in d['cores']:
        if core['core'] is not None:
            r=requests.get(f"{API}/cores/{core['core']}").json(); Block.append(r['block']); ReusedCount.append(r['reuse_count']); Serial.append(r['serial'])
        else: Block.append(None); ReusedCount.append(None); Serial.append(None)
        Outcome.append(f"{core['landing_success']} {core['landing_type']}"); Flights.append(core['flight']); GridFins.append(core['gridfins'])
        Reused.append(core['reused']); Legs.append(core['legs']); LandingPad.append(core['landpad'])

In [4]:
try:
    for f in (getBoosterVersion, getLaunchSite, getPayloadData, getCoreData): f(data)
    launch = pd.DataFrame({'FlightNumber':list(data['flight_number']),'Date':list(data['date']),'BoosterVersion':BoosterVersion,
        'PayloadMass':PayloadMass,'Orbit':Orbit,'LaunchSite':LaunchSite,'Outcome':Outcome,'Flights':Flights,'GridFins':GridFins,
        'Reused':Reused,'Legs':Legs,'LandingPad':LandingPad,'Block':Block,'ReusedCount':ReusedCount,'Serial':Serial,
        'Longitude':Longitude,'Latitude':Latitude})
    data_falcon9 = launch[launch['BoosterVersion']!='Falcon 1'].copy()
    data_falcon9['FlightNumber'] = range(1, len(data_falcon9)+1)
except Exception as e:
    print('Live API unavailable, using IBM snapshot:', e)
    data_falcon9 = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv')
data_falcon9.isnull().sum()

Live API unavailable, using IBM snapshot: Expecting value: line 1 column 1 (char 0)


FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

In [5]:
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].fillna(data_falcon9['PayloadMass'].mean())
data_falcon9.to_csv('dataset_part_1.csv', index=False)
print(data_falcon9.shape); data_falcon9.head()

(90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
